In [0]:
from pyspark.sql.functions import col, lit, date_format, to_date, sequence, explode, min, max, when, expr, dayofweek, weekofyear

DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/" 
SILVER_BASE_PATH = f"{DELTA_VOLUME_PATH}silver/" 
GOLD_BASE_PATH = f"{DELTA_VOLUME_PATH}gold/" 

print("Starting Gold Layer Modeling...")

# Load Silver Tables
df_stoptimes = spark.read.format("delta").load(f"{SILVER_BASE_PATH}stop_times/")
df_trips = spark.read.format("delta").load(f"{SILVER_BASE_PATH}trips/")
df_routes = spark.read.format("delta").load(f"{SILVER_BASE_PATH}routes/")
df_stops = spark.read.format("delta").load(f"{SILVER_BASE_PATH}stops/")
df_calendar = spark.read.format("delta").load(f"{SILVER_BASE_PATH}calendar/")

In [0]:
df_fact = df_stoptimes.alias("st") \
            .join(
                df_trips.alias("t"),
                on = (col("st.trip_id") == col("t.trip_id")),
                how = "inner"
            ) \
            .select(
                col("st.*"),
                col("t.route_id"),
                col("t.service_id"),
                col("t.direction_id"),
                col("t.trip_headsign")
            )


df_fact = df_fact.alias("f") \
        .join(
            df_routes.alias("r"),
            on = (col("f.route_id") == col("r.route_id")),
            how = "left"
        ) \
        .select(
            col("f.*"),
            col("r.route_short_name_display"),
            col("r.route_long_name"),
            col("r.route_type")
        )

df_fact = df_fact.alias("f") \
        .join(
            df_stops.alias("s"),
            on = (col("f.stop_id") == col("s.stop_id")),
            how = "left"
        ) \
        .select(
            col("f.*"),
            col("s.stop_name_detail"),
            col("s.stop_lat"),
            col("s.stop_lon")
        )
    
df_fact = df_fact.alias("f") \
        .join(
            df_calendar.alias("c"),
            on = (col("f.service_id") == col("c.service_id")),
            how = "left"
        ) \
        .select(
            col("f.*"),
            col("c.monday"),
            col("c.tuesday"),
            col("c.wednesday"),
            col("c.thursday"),
            col("c.friday"),
            col("c.saturday"),
            col("c.sunday"),
            col("c.start_date"),
            col("c.end_date")
        )

print(f"Created Gold Fact Table")


In [0]:
df_fact.display(5)

In [0]:
gold_fact_path = f"{GOLD_BASE_PATH}fact_trip_schedule/"

(df_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") 
    .save(gold_fact_path)
)

row_count = spark.read.format("delta").load(gold_fact_path).count()
print(f"Successfully saved Gold Fact Table with {row_count} rows to: {gold_fact_path}")

In [0]:
# Optimizing

"""
The Gold Fact table is the batch side of the Stream-to-Batch Join. Since Spark must repeatedly scan this batch table to enrich every incoming micro-batch of stream data, its read performance is critical. I used Z-Ordering to co-locate data points with similar trip_id and route_id values on disk
"""
spark.sql(f"""
    OPTIMIZE delta.`{gold_fact_path}`
    ZORDER BY (trip_id, route_id)
""")

In [0]:
df_fact.display(5)

In [0]:
print("Creating dim_date table...")

df_calendar = spark.read.format("delta").load(f"{SILVER_BASE_PATH}calendar/")


# Get minimum start and end date 
min_date_str = (df_calendar
    .select(min("start_date"))
    .collect()[0][0] 
)
max_date_str = (df_calendar
    .select(max("end_date"))
    .collect()[0][0] # Access the first row [0] and the first column [0]
)

# 2. Generate all dates between the min and max
# Convert the collected Python date objects back into Spark SQL DateType for sequence()
min_date = to_date(lit(str(min_date_str)), "yyyy-MM-dd")
max_date = to_date(lit(str(max_date_str)), "yyyy-MM-dd")

df_date_spine = spark.range(1) \
    .select(explode(sequence(min_date, max_date, expr("INTERVAL 1 DAY"))).alias("date")) \
    .select(col("date").cast("date"))


# 3. Add dimension columns 
df_dim_date = df_date_spine.select(
    col("date"),
    date_format(col("date"), "yyyyMMdd").cast("integer").alias("date_key"),
    date_format(col("date"), "yyyy").cast("integer").alias("year"),
    date_format(col("date"), "MM").cast("integer").alias("month"),
    date_format(col("date"), "dd").cast("integer").alias("day"),
    date_format(col("date"), "EEEE").alias("day_name"),
    ((dayofweek(col("date")) + 5) % 7 + 1).alias("day_of_week"), 
    weekofyear(col("date")).alias("week_of_year"),
    when(((dayofweek(col("date")) + 5) % 7 + 1).isin(6, 7), lit(True)).otherwise(lit(False)).alias("is_weekend")
)

# 4. Write the Dimension Table
gold_dim_date_path = f"{GOLD_BASE_PATH}dim_date/"

(df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") 
    .save(gold_dim_date_path)
)

print(f"Successfully saved Gold Dimension Table: dim_date.")

In [0]:
df_dim_date.display()

In [0]:
# Register gold tables in unity catalog

CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "gold"

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

gold_tables_to_register = ["fact_trip_schedule", "dim_date"]

for table in gold_tables_to_register:
    path = f"{GOLD_BASE_PATH}{table}/"
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table}"

    df_to_register = spark.read.format("delta").load(path)

    (df_to_register.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") 
        .saveAsTable(full_table_name)
    )
    print(f"Registered Managed Table: {full_table_name}")
    
print("\nGold Layer Processing and Managed Registration Complete.")